# Lab 5 Agent Look
## MM redo only

In [4]:
# imports
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

openai = OpenAI()

In [5]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
todos = []
completed = []

In [9]:
# fn to dispay and return the todo report
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [10]:
get_todo_report()

todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [13]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [14]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [17]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [18]:
todos, completed = [], []
loop(messages)

Todo #1: Interpret the problem and set up variables (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance) with a reasonable value.
Todo #3: Compute each train’s travel time and solve for the meeting time.
Todo #4: Convert the result to a clock time and present the final answer clearly.

Let D be the distance between Boston and New York along the track. Let t be the number of hours after 2:00 pm when 
they meet.

Boston train distance: 60t.

NY train leaves 1 hour later, so it travels for (t-1) hours by meeting time, covering 80(t-1).

They meet when: 60t + 80(t-1) = D.

Todo #1: Interpret the problem and set up variables (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance) with a reasonable value.
Todo #3: Compute each train’s travel time and solve for the meeting time.
Todo #4: Convert the result to a clock time and present the final answer clearly.

The problem omits D. A common approximate rail/highway distance Boston3New York is about 215 miles (NYC to Boston 
is typically ~21520 miles).

Use D = 215 mi as a reasonable estimate.

Todo #1: Interpret the problem and set up variables (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance) with a reasonable value.
Todo #3: Compute each train’s travel time and solve for the meeting time.
Todo #4: Convert the result to a clock time and present the final answer clearly.

Solve 60t + 80(t-1) = D.

60t + 80t - 80 = D => 140t = D + 80 => t = (D+80)/140.

With D=215: t = (215+80)/140 = 295/140 = 2.107142... hours after 2:00 pm.

That is 2 hours + 0.107142*60 minutes = 2 hours + 6.4285 minutes => about 2 hours 6 minutes 26 seconds.

Todo #1: Interpret the problem and set up variables (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance) with a reasonable value.
Todo #3: Compute each train’s travel time and solve for the meeting time.
Todo #4: Convert the result to a clock time and present the final answer clearly.

Meeting time = 2:00 pm + 2:06:26 => approximately 4:06 pm (about 4:06:26 pm) assuming D = 215 miles.

General form (if exact D is provided): meeting time is 2:00 pm + (D+80)/140 hours.

Todo #1: Interpret the problem and set up variables (distance between Boston and New York).
Todo #2: Estimate any missing quantity (Boston–New York distance) with a reasonable value.
Todo #3: Compute each train’s travel time and solve for the meeting time.
Todo #4: Convert the result to a clock time and present the final answer clearly.

Let D be the Boston–New York distance (miles) and let t be hours after 2:00 pm when they meet.

Boston train travels: 60t  
NY train leaves at 3:00 pm, so it travels: 80(t−1)

They meet when:
60t + 80(t−1) = D  
140t = D + 80  
t = (D + 80)/140

Using a reasonable estimate D ≈ 215 miles:
t = (215+80)/140 = 295/140 ≈ 2.107 hours  
That’s about 2 hours 6 minutes after 2:00 pm → about 4:06 pm (≈ 4:06:26 pm).